# CDRv8 srWGS Known Issues #10

In [ ]:
import pandas as pd

In [ ]:
! gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/known_issues/rids_with_low_coverage_all_failures_wgs_v8_known_issue_10.tsv ./wgs/v8_srWGS_known_qc_issues/

In [ ]:
df_know_issues = pd.read_csv('rids_with_low_coverage_all_failures_wgs_v8_known_issue_10.tsv', sep='\t')
df_know_issues.columns = ['IID']

In [ ]:
df_cohort = pd.read_csv('cohort_covariate.txt', sep='\t')
iids_to_remove = df_know_issues['IID'].tolist()
df_cohort_filtered = df_cohort[~df_cohort['IID'].isin(iids_to_remove)]

df_cohort_filtered.to_csv('cohort_covariate_KnownIssues_REMOVED.txt', sep='\t', index=False)

print(df_cohort_filtered)

# Sex Check

In [ ]:
! gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv ./wgs/qc/

In [ ]:
wgs_qc_metrics = pd.read_csv('wgs/qc/genomic_metrics.tsv', sep='\t')
# wgs_qc_metrics.value_counts('dragen_sex_ploidy')

In [ ]:
aou_v8_all_with_CNV_df = pd.read_csv('covariate_files/aou_v8_all_participants_with_CNV_data.txt', sep='\t')

In [ ]:
aou_v8_all_with_CNV_df_ids = aou_v8_all_with_CNV_df['IID'].tolist()
wgs_qc_metrics = wgs_qc_metrics[wgs_qc_metrics['research_id'].isin(aou_v8_all_with_CNV_df_ids)]

aou_v8_all_with_CNV_df_filtered = aou_v8_all_with_CNV_df[aou_v8_all_with_CNV_df['sex_at_birth'].isin(['Male', 'Female'])]

In [ ]:
wgs_qc_metrics_id_sex = wgs_qc_metrics[['research_id', 'dragen_sex_ploidy']]
# Keeping only XX and XY Dragen Ploidy
wgs_qc_metrics_id_sex = wgs_qc_metrics_id_sex[wgs_qc_metrics_id_sex['dragen_sex_ploidy'].isin(['XX', 'XY'])]

wgs_qc_metrics_id_sex.columns = ['IID', 'dragen_sex_ploidy']

iids_to_keep = wgs_qc_metrics_id_sex['IID'].tolist()
aou_v8_all_with_CNV_df_filtered_sex_concordance = aou_v8_all_with_CNV_df_filtered[aou_v8_all_with_CNV_df_filtered['IID'].isin(iids_to_keep)]

aou_v8_all_with_CNV_df_filtered_sex_concordance.to_csv('covariate_files/aou_v8_all_participants_with_CNV_data_Sex_checked.txt', sep='\t', index=False)

In [ ]:
iids_to_keep = aou_v8_all_with_CNV_df_filtered_sex_concordance['IID'].tolist()
df_cohort_filtered_sex_check = df_cohort_filtered[df_cohort_filtered['IID'].isin(iids_to_keep)]
df_cohort_filtered_sex_check.shape

df_cohort_all_covariates_pheno    = pd.read_csv('cohort_covariate_all_covariates_pheno.txt', sep='\t')
df_cohort_filtered_sex_check_pheno = pd.merge(df_cohort_filtered_sex_check, df_cohort_all_covariates_pheno[['IID', 'AUD']], how='left', left_on='IID', right_on='IID')
df_cohort_filtered_sex_check_pheno.to_csv('cohort_covariate_KnownIssues_REMOVED_SexChecked.txt', sep='\t', index=False)

wgs_qc_metrics_sex = wgs_qc_metrics[wgs_qc_metrics['dragen_sex_ploidy'].isin(['XX','XY'])]


In [ ]:
# Step 1: Create a mapping dictionary for sex_at_birth
# 1 -> XY, 2 -> XX
sex_mapping = {1: 'XY', 2: 'XX'}

# Step 2: Create a new column in the first dataframe for direct comparison
df_cohort_filtered_sex_check_pheno['mapped_sex'] = df_cohort_filtered_sex_check_pheno['sex_at_birth'].map(sex_mapping)

# Step 3: Merge the two dataframes on the common IID column
# This assumes the IID column is named 'IID' in both dataframes
merged_df = pd.merge(
    df_cohort_filtered_sex_check_pheno,
    wgs_qc_metrics_sex,
    left_on='IID',
    right_on = 'research_id', # Change 'IID' to the actual name of your IID column
    how='inner'
)

# Step 4: Find the mismatches
# Compare the 'mapped_sex' column with the 'dragen_sex_ploidy' column
mismatches_df = merged_df[merged_df['mapped_sex'] != merged_df['dragen_sex_ploidy']]

# Display the IIDs with mismatches
# You can see the original values to understand the mismatch
print("\nIIDs with sex mismatches:")
print(mismatches_df[['IID', 'mapped_sex', 'dragen_sex_ploidy']])

# To get the count of mismatches
print("\nNumber of mismatches:", len(mismatches_df))

# Relatedness (not recommended)

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv ./wgs/relatedness/

In [ ]:
df_aou_v8_cnv_filtered = pd.read_csv('covariate_files/aou_v8_all_participants_with_CNV_data_Sex_checked.txt', sep = '\t')
df_relatedness_flagged = pd.read_csv('wgs/relatedness/relatedness_flagged_samples.tsv', sep='\t')

In [ ]:
df_aou_v8_cnv_filtered_relatedness = df_aou_v8_cnv_filtered[~df_aou_v8_cnv_filtered['IID'].isin(df_relatedness_flagged['sample_id'])]
df_aou_v8_cnv_filtered_relatedness.to_csv('covariate_files/aou_v8_all_participants_with_CNV_data_Sex_checked_Relatedness_removed.txt', sep='\t', index=False)

In [ ]:
df_cohort_filtered_sex_check_pheno_relatedness = df_cohort_filtered_sex_check_pheno[~df_cohort_filtered_sex_check_pheno['IID'].isin(df_relatedness_flagged['sample_id'])]
df_cohort_filtered_sex_check_pheno_relatedness.to_csv('cohort_covariate_KnownIssues_REMOVED_SexChecked_Relatedness_REMOVED.txt', sep = '\t', index=False)

# SV Minimal set of related samples to prune (This is Final Cohort)

In [ ]:
! gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/structural_variants/aux/relatedness/AoU_srWGS_SV.v8.minimal_set_of_related_samples_to_prune.txt ./wgs/relatedness/

In [ ]:
df_sv_relatedness = pd.read_csv('wgs/relatedness/AoU_srWGS_SV.v8.minimal_set_of_related_samples_to_prune.txt', sep='\t',header=None)
df_sv_relatedness.rename(columns={0: 'IID'}, inplace=True)

df_sv_relatedness = df_sv_relatedness['IID'].tolist()
df_aud_wgsqc_sex = df_cohort_filtered_sex_check_pheno[~df_cohort_filtered_sex_check_pheno['IID'].isin(df_sv_relatedness)]

In [ ]:
df_aud_wgsqc_sex.to_csv('aud/AUD_Cases_Controls_EHR_Depth/all_covariates_AUD_EHR_Depth_age_sex_pcs_updated_20250729_KnownIssues_REMOVED_SexChecked_SV_PrunedRelatedness_REMOVED.txt', 
                        sep='\t', index=False)

# END